# Practical PyTorch · Part 13 — Companion Notebook

This goes with **"Beyond pipeline(): Taking the Wheel with from_pretrained"**. Run the cells top to bottom (Shift+Enter).

We drop one level below `pipeline()` and run the three steps ourselves — **preprocess → forward → postprocess** — so we can get at the raw scores.

## 0. Setup

PyTorch is preinstalled on Colab; we just add `transformers`.

In [ ]:
!pip install -q transformers

## 1. Load the matched pair: a tokenizer and a model

Both come from the **same name** via `from_pretrained`. The tokenizer turns words into the exact integer IDs this model expects; the `...ForSequenceClassification` class puts a classifier head on top.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "distilbert-base-uncased-finetuned-sst-2-english"

tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)

print("loaded:", name)
print("labels:", model.config.id2label)

## 2. Preprocess — text to tensors

`return_tensors="pt"` is the part everyone forgets. "pt" means PyTorch; without it you get plain Python lists and the model refuses them.

In [ ]:
inputs = tok("I love this", return_tensors="pt")
print(inputs)
# input_ids  -> your sentence as integers the model knows
# attention_mask -> which positions are real words vs padding

## 3. Forward — run the tensors through

`**inputs` unpacks the dict into keyword arguments. `torch.no_grad()` skips the training-only gradient bookkeeping — faster and lighter, no downside for inference.

In [ ]:
import torch

with torch.no_grad():
    logits = model(**inputs).logits

print("logits:", logits)        # raw, unnormalized scores
print("shape: ", logits.shape)  # [batch, num_labels] -> [1, 2]

## 4. Postprocess — logits to a human answer

`softmax` turns raw logits into probabilities that sum to 1. `id2label` maps the winning index to its name — never hard-code the label, the model tells you.

In [ ]:
probs = logits.softmax(dim=-1)
print("probs:", probs)

winner = probs.argmax().item()
print("label:", model.config.id2label[winner])
print("confidence:", round(probs[0, winner].item(), 4))

# The whole point of dropping down: every class score is right here.
for i, p in enumerate(probs[0]):
    print(f"  {model.config.id2label[i]}: {p.item():.4f}")

## 5. The whole recipe, batched

The same three steps handle many inputs at once. `padding=True` lets the tokenizer line up sentences of different lengths into one tensor. Now the batch dimension is the number of sentences.

In [ ]:
texts = [
    "I can't believe how easy this was.",
    "Setting up a GPU used to ruin my whole afternoon.",
    "It's fine, I guess.",
]

inputs = tok(texts, return_tensors="pt", padding=True)   # preprocess
with torch.no_grad():
    logits = model(**inputs).logits                      # forward
probs = logits.softmax(dim=-1)                           # postprocess

for text, row in zip(texts, probs):
    label = model.config.id2label[row.argmax().item()]
    print(f"{label:>8}  ({row.max().item():.3f})  {text}")

## 6. The same recipe for vision (brief)

Images follow the identical pattern — only the preprocessor's name changes. Instead of a tokenizer you use an **image processor** that resizes and normalizes the picture. The forward and postprocess steps are word-for-word the same.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import requests

name = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(name)
vmodel = AutoModelForImageClassification.from_pretrained(name)

url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # two cats
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(images=image, return_tensors="pt")   # preprocess
with torch.no_grad():
    logits = vmodel(**inputs).logits                    # forward
label = vmodel.config.id2label[logits.argmax(-1).item()]  # postprocess
print("prediction:", label)

Same shape every time: **preprocess → forward → postprocess.** `pipeline()` did all three for you; now you can do them by hand whenever you need the control. Next up (Part 14): embeddings — turning inputs into vectors you can search and compare.